# LeetCode 3165 — Maximum Sum of Subsequence With Non-adjacent Elements bằng Python

- Link: https://leetcode.com/problems/maximum-sum-of-subsequence-with-non-adjacent-elements/
- Độ khó: **Hard**
- Chủ đề: **Segment tree lưu 4 trạng thái biên** (được/không được chọn phần tử đầu, cuối đoạn) để trả lời bài "House Robber" động với cập nhật điểm; tương đương **tích ma trận (max, +)** 2×2

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maximumSumSubsequence(self, nums: List[int], queries: List[List[int]]) -> int:
```

## 1. Mục tiêu học

1. Nhớ lại House Robber: tổng lớn nhất các phần tử không kề nhau (được chọn rỗng ⇒ đáp án `≥ 0`).
2. Thấy DP tuyến tính không hợp với **cập nhật điểm** (mỗi truy vấn `O(n)`).
3. Thiết kế thông tin **ghép được** cho một đoạn: `v[a][b]` = tổng tốt nhất khi phần tử **đầu** được phép chọn (`a = 1`) hay bị cấm (`a = 0`), tương tự phần tử **cuối** (`b`).
4. Ghép hai đoạn kề `L | R`: không được chọn đồng thời cuối `L` và đầu `R` ⇒ `res[a][b] = max(L[a][1] + R[0][b], L[a][0] + R[1][b])`.
5. Segment tree cập nhật `O(log n)`, đáp án là `root[1][1]`.

## 2. Đề bài & ví dụ

Cho mảng `nums` và các truy vấn `[pos, x]`. Với mỗi truy vấn: gán `nums[pos] = x`, rồi tính tổng lớn nhất của một dãy con mà **không có hai phần tử kề nhau** (trong mảng) cùng được chọn (dãy rỗng có tổng 0). Trả về **tổng các đáp án** mod `10^9 + 7`.

**Ví dụ 1:** `nums = [3,5,9]`, `queries = [[1,-2],[0,-3]]` → `21` (`12 + 9`).

**Ví dụ 2:** `nums = [0,-1]`, `queries = [[0,-5]]` → `0`.

**Ràng buộc:** `1 ≤ n, q ≤ 5·10⁴`; `−10⁵ ≤ nums[i], x ≤ 10⁵`.

In [1]:
from typing import List
import random, time

MOD = 10 ** 9 + 7
examples = [([3, 5, 9], [[1, -2], [0, -3]], 21), ([0, -1], [[0, -5]], 0)]
for a, q, exp in examples:
    print(a, q, '-> kỳ vọng', exp)

[3, 5, 9] [[1, -2], [0, -3]] -> kỳ vọng 21
[0, -1] [[0, -5]] -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1. House Robber `O(n)`

`take` = tốt nhất khi chọn phần tử hiện tại, `skip` = tốt nhất khi không chọn: `take' = skip + x`, `skip' = max(take, skip)`. Đáp án `max(take, skip)` (khởi đầu `0, 0` cho phép dãy rỗng).

In [2]:
def robber(a):
    take = skip = 0
    for x in a:
        take, skip = skip + x, max(take, skip)
    return max(take, skip)

print(robber([3, -2, 9]), robber([-3, -2, 9]), robber([-5, -1]), robber([2, 7, 9, 3, 1]))

12 9 0 12


### 3.2. Thông tin ghép được của một đoạn

Với đoạn `S`, định nghĩa `v[a][b]` (`a, b ∈ {0, 1}`) = tổng lớn nhất của một tập không kề nhau trong `S` sao cho: nếu `a = 0` thì **không** chọn phần tử đầu `S`; nếu `b = 0` thì **không** chọn phần tử cuối. (`a = 1`/`b = 1` nghĩa là "được phép", không bắt buộc.)

- Lá `x`: `v[1][1] = max(0, x)`, các ô còn lại `0`.
- Ghép `L | R`: phần tử cuối `L` và đầu `R` kề nhau ⇒ ít nhất một trong hai bị cấm:
  `res[a][b] = max(L[a][1] + R[0][b], L[a][0] + R[1][b])`.

In [3]:
def leaf(x):
    m = max(0, x)
    return (0, 0, 0, m)                 # (v00, v01, v10, v11)

def merge(L, R):
    l00, l01, l10, l11 = L; r00, r01, r10, r11 = R
    return (max(l01 + r00, l00 + r10),  # a=0, b=0
            max(l01 + r01, l00 + r11),  # a=0, b=1
            max(l11 + r00, l10 + r10),  # a=1, b=0
            max(l11 + r01, l10 + r11))  # a=1, b=1

from functools import reduce
for a in ([3, -2, 9], [2, 7, 9, 3, 1], [5, 1, 1, 5]):
    print(a, '-> ghép các lá:', reduce(merge, map(leaf, a))[3], '| House Robber:', robber(a))

[3, -2, 9] -> ghép các lá: 12 | House Robber: 12
[2, 7, 9, 3, 1] -> ghép các lá: 12 | House Robber: 12
[5, 1, 1, 5] -> ghép các lá: 10 | House Robber: 10


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Khi chỉ một phần tử đổi, phần lớn mảng "giữ nguyên câu trả lời con". Chia mảng thành cây đoạn; mỗi nút lưu 4 số mô tả đoạn của nó **kèm thông tin biên** — đủ để ghép với hàng xóm mà không vi phạm "không kề". Cập nhật một lá ⇒ tính lại các tổ tiên (`log n` nút).

Ví dụ 1 sau truy vấn 1 (`[3, −2, 9]`): ghép `3 | −2 | 9` cho `v[1][1] = 12`.

## 5. Cách 1 — Brute Force: chạy House Robber lại sau mỗi truy vấn

In [4]:
def brute(nums, queries):
    a = list(nums); total = 0
    for p, x in queries:
        a[p] = x
        total += robber(a)
    return total % MOD

for a, q, exp in examples:
    print('brute:', brute(a, q), '| kỳ vọng', exp)

brute: 21 | kỳ vọng 21
brute: 0 | kỳ vọng 0


**Độ phức tạp:** `O(n · q)` = `2.5·10⁹` xấu nhất — quá chậm.

## 6. Ý tưởng thuật toán / bước đột phá

```
dựng segment tree, mỗi nút = (v00, v01, v10, v11)
for (p, x) in queries:
    đặt lá p = leaf(x); đi lên: nút = merge(con trái, con phải)
    ans += root.v11
```

Mỗi truy vấn `O(log n)` phép `merge` (8 phép cộng + 4 `max`).

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Segment tree bottom-up, 4 mảng (lời giải chính)

Kích thước lá làm tròn lên lũy thừa 2; các lá đệm có giá trị `0` (thêm phần tử 0 ở **cuối** mảng không đổi đáp án). Vì phép ghép **không giao hoán**, luôn ghép `con trái` rồi `con phải`.

In [5]:
def segtree_solution(nums, queries):
    n = len(nums); size = 1
    while size < n:
        size *= 2
    A = [0] * (2 * size); B = [0] * (2 * size); C = [0] * (2 * size); D = [0] * (2 * size)   # v00 v01 v10 v11
    for i, x in enumerate(nums):
        D[size + i] = x if x > 0 else 0
    def pull(i):
        l, r = 2 * i, 2 * i + 1
        A[i] = max(B[l] + A[r], A[l] + C[r]); B[i] = max(B[l] + B[r], A[l] + D[r])
        C[i] = max(D[l] + A[r], C[l] + C[r]); D[i] = max(D[l] + B[r], C[l] + D[r])
    for i in range(size - 1, 0, -1):
        pull(i)
    total = 0
    for p, x in queries:
        i = size + p; D[i] = x if x > 0 else 0; i //= 2
        while i:
            pull(i); i //= 2
        total += D[1]
    return total % MOD

for a, q, exp in examples:
    print('segment tree:', segtree_solution(a, q), '| kỳ vọng', exp)

segment tree: 21 | kỳ vọng 21
segment tree: 0 | kỳ vọng 0


### 7.2. Cách 3 — Ma trận (max, +) 2×2

Viết House Robber dạng tuyến tính trên nửa vành `(max, +)`: trạng thái `(skip, take)`, phần tử `x` có ma trận `M_x = [[0, 0], [x, −∞]]` sao cho `(skip', take') = M_x ⊗ (skip, take)`. Tích các ma trận theo thứ tự là phép **kết hợp** ⇒ segment tree trên ma trận. (Tương đương 4 trạng thái biên — chỉ là cách nhìn khác.)

In [6]:
NEG = float('-inf')
def mat(x):
    return ((0, 0), (x, NEG))           # skip' = max(skip, take); take' = skip + x

def mul(P, Q):                          # (P ⊗ Q)[i][j] = max_k P[i][k] + Q[k][j]
    return tuple(tuple(max(P[i][k] + Q[k][j] for k in range(2)) for j in range(2)) for i in range(2))

def maxplus_solution(nums, queries):
    n = len(nums); size = 1
    while size < n:
        size *= 2
    I = ((0, NEG), (NEG, 0))
    T = [I] * (2 * size)
    for i, x in enumerate(nums):
        T[size + i] = mat(x)
    for i in range(size - 1, 0, -1):
        T[i] = mul(T[2 * i + 1], T[2 * i])     # áp phần tử trái trước: M_r ⊗ M_l
    total = 0
    for p, x in queries:
        i = size + p; T[i] = mat(x); i //= 2
        while i:
            T[i] = mul(T[2 * i + 1], T[2 * i]); i //= 2
        M = T[1]
        total += max(max(M[0][0], M[1][0]), 0)  # trạng thái đầu (skip, take) = (0, -inf)
    return total % MOD

for a, q, exp in examples:
    print('ma trận (max,+):', maxplus_solution(a, q), '| kỳ vọng', exp)

ma trận (max,+): 21 | kỳ vọng 21
ma trận (max,+): 0 | kỳ vọng 0


### 7.3. Cách 4 — Chia căn: mỗi khối lưu 4 trạng thái biên

Khối cỡ `B ≈ √n`; cập nhật một phần tử ⇒ tính lại khối của nó `O(B)`; trả lời ⇒ ghép `n / B` khối `O(n / B)`. Tổng `O(q √n)`.

In [7]:
def sqrt_solution(nums, queries):
    a = list(nums); n = len(a); Bsz = max(1, int(n ** 0.5))
    blocks = [None] * ((n + Bsz - 1) // Bsz)
    def rebuild(b):
        blocks[b] = reduce(merge, (leaf(x) for x in a[b * Bsz: (b + 1) * Bsz]))
    for b in range(len(blocks)):
        rebuild(b)
    total = 0
    for p, x in queries:
        a[p] = x; rebuild(p // Bsz)
        total += reduce(merge, blocks)[3]
    return total % MOD

for a, q, exp in examples:
    print('chia căn:', sqrt_solution(a, q), '| kỳ vọng', exp)

chia căn: 21 | kỳ vọng 21
chia căn: 0 | kỳ vọng 0


## 8. TRACE từng bước

Trace cách 2 trên `nums = [4, 1, 1, 4, 2]` với vài truy vấn: in `root = (v00, v01, v10, v11)`.

In [8]:
def trace(nums, queries):
    a = list(nums)
    for p, x in queries:
        a[p] = x
        root = reduce(merge, map(leaf, a))
        print(f'gán nums[{p}] = {x:3d}: nums = {a} | root (v00, v01, v10, v11) = {root} | đáp án {root[3]} | robber {robber(a)}')
    print('tổng:', segtree_solution(nums, queries), '| brute:', brute(nums, queries))

trace([4, 1, 1, 4, 2], [[2, 10], [0, -3], [4, 7], [2, -1]])

gán nums[2] =  10: nums = [4, 1, 10, 4, 2] | root (v00, v01, v10, v11) = (10, 12, 14, 16) | đáp án 16 | robber 16
gán nums[0] =  -3: nums = [-3, 1, 10, 4, 2] | root (v00, v01, v10, v11) = (10, 12, 10, 12) | đáp án 12 | robber 12
gán nums[4] =   7: nums = [-3, 1, 10, 4, 7] | root (v00, v01, v10, v11) = (10, 17, 10, 17) | đáp án 17 | robber 17
gán nums[2] =  -1: nums = [-3, 1, -1, 4, 7] | root (v00, v01, v10, v11) = (5, 8, 5, 8) | đáp án 8 | robber 8
tổng: 53 | brute: 53


## 9. Code nộp LeetCode

In [9]:
class Solution:
    def maximumSumSubsequence(self, nums: List[int], queries: List[List[int]]) -> int:
        n = len(nums)
        size = 1
        while size < n:
            size *= 2
        A = [0] * (2 * size)
        B = [0] * (2 * size)
        C = [0] * (2 * size)
        D = [0] * (2 * size)
        for i, x in enumerate(nums):
            if x > 0:
                D[size + i] = x
        for i in range(size - 1, 0, -1):
            l = 2 * i
            r = l + 1
            A[i] = max(B[l] + A[r], A[l] + C[r])
            B[i] = max(B[l] + B[r], A[l] + D[r])
            C[i] = max(D[l] + A[r], C[l] + C[r])
            D[i] = max(D[l] + B[r], C[l] + D[r])
        total = 0
        for p, x in queries:
            i = size + p
            D[i] = x if x > 0 else 0
            i >>= 1
            while i:
                l = 2 * i
                r = l + 1
                A[i] = max(B[l] + A[r], A[l] + C[r])
                B[i] = max(B[l] + B[r], A[l] + D[r])
                C[i] = max(D[l] + A[r], C[l] + C[r])
                D[i] = max(D[l] + B[r], C[l] + D[r])
                i >>= 1
            total += D[1]
        return total % (10 ** 9 + 7)

for a, q, exp in examples:
    got = Solution().maximumSumSubsequence(list(a), q)
    print(got, 'OK' if got == exp else 'SAI')

21 OK
0 OK


## 10. Vì sao thuật toán đúng

1. **Lá:** đoạn một phần tử `x`: nếu phần tử bị cấm (ở đầu hoặc cuối — chính là nó) thì chỉ có tập rỗng ⇒ 0; nếu được phép cả hai đầu ⇒ `max(0, x)`.
2. **Ghép:** một tập hợp lệ trên `L | R` là hợp của tập hợp lệ trên `L` và trên `R` sao cho không chọn đồng thời cuối `L` và đầu `R`. Chia hai trường hợp: cấm đầu `R` (`L[a][1] + R[0][b]`) hoặc cấm cuối `L` (`L[a][0] + R[1][b]`); hai trường hợp phủ mọi tập hợp lệ và mỗi tập trong đó hợp lệ ⇒ `max` đúng.
3. **Đệm:** các lá đệm `0` ở cuối không tạo tập hợp lệ tốt hơn (chọn 0 không đổi tổng) ⇒ `root[1][1]` đúng đáp án.
4. **Mod:** mỗi đáp án `≥ 0`, cộng dồn bằng số nguyên Python rồi mod một lần cuối — tương đương mod từng bước.

## 11. Phân tích Time / Space Complexity

| Cách | Dựng | Mỗi truy vấn | Space |
|---|---|---|---|
| 1. House Robber lại | — | `O(n)` | `O(n)` |
| 2. Segment tree 4 mảng | `O(n)` | `O(log n)` | `O(n)` |
| 3. Ma trận (max, +) | `O(n)` | `O(8 log n)` | `O(n)` |
| 4. Chia căn | `O(n)` | `O(√n)` | `O(n)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [10]:
impls = {'segtree': segtree_solution, 'maxplus': maxplus_solution, 'sqrt': sqrt_solution,
         'submit': lambda a, q: Solution().maximumSumSubsequence(list(a), q)}
edge = [([5], [[0, 7], [0, -1], [0, 3]], 10), ([-1, -2, -3], [[1, -5]], 0), ([1, 1], [[0, 2], [1, 3]], 5),
        ([1, 2, 3, 4], [[0, 10], [3, 10]], 34), ([5, 5, 5, 5, 5], [[2, 0]], 10)]
for a, q, exp in examples + edge:
    assert brute(a, q) == exp, (a, q, brute(a, q))
    for name, f in impls.items():
        assert f(a, q) == exp, (name, a, q)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


Giải thích: `[1, 2, 3, 4]` → sau `nums[0] = 10`: `[10, 2, 3, 4]` ⇒ tốt nhất `10 + 4 = 14`; sau `nums[3] = 10`: `[10, 2, 3, 10]` ⇒ `20` ⇒ tổng `34`. `[5, 5, 5, 5, 5]` với `nums[2] = 0` ⇒ `[5, 5, 0, 5, 5]`: mọi tập không kề nhau chứa nhiều nhất hai số 5 (ví dụ chỉ số 0 và 3) ⇒ `10`.

### 12.2. Đối chiếu ngẫu nhiên với brute

In [11]:
rng = random.Random(3165)
cnt = 0
for _ in range(300):
    n = rng.randint(1, 12); a = [rng.randint(-6, 6) for _ in range(n)]
    q = [[rng.randint(0, n - 1), rng.randint(-6, 6)] for _ in range(rng.randint(1, 10))]
    exp = brute(a, q)
    for name, f in impls.items():
        assert f(a, q) == exp, (name, a, q)
    cnt += 1
print('Khớp brute trên', cnt, 'bộ ngẫu nhiên.')

Khớp brute trên 300 bộ ngẫu nhiên.


### 12.3. Ca lớn `n = q = 5·10⁴`

In [12]:
N = 5 * 10 ** 4
cases = {
    'ngẫu nhiên': ([rng.randint(-10 ** 5, 10 ** 5) for _ in range(N)],
                   [[rng.randint(0, N - 1), rng.randint(-10 ** 5, 10 ** 5)] for _ in range(N)]),
    'toàn dương': ([10 ** 5] * N, [[rng.randint(0, N - 1), 10 ** 5] for _ in range(N)]),
}
for name, (a, q) in cases.items():
    t = time.time(); x = Solution().maximumSumSubsequence(list(a), q); t1 = time.time() - t
    t = time.time(); y = segtree_solution(a, q); t2 = time.time() - t
    assert x == y
    print(f'{name:12s}: {x:12d} | nộp {t1:.2f}s | segtree hàm pull {t2:.2f}s')
assert Solution().maximumSumSubsequence([10 ** 5] * N, [[0, 10 ** 5]]) == (N + 1) // 2 * 10 ** 5 % MOD
print('Toàn dương bằng nhau: chọn ⌈N/2⌉ phần tử: đúng.')

ngẫu nhiên  :    530206220 | nộp 0.73s | segtree hàm pull 0.65s


toàn dương  :    999125007 | nộp 0.86s | segtree hàm pull 0.81s
Toàn dương bằng nhau: chọn ⌈N/2⌉ phần tử: đúng.


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Bắt buộc chọn ít nhất một phần tử (lá `v11 = x` thay vì `max(0, x)`)

In [13]:
def wrong_no_empty(nums, queries):
    a = list(nums); total = 0
    for p, x in queries:
        a[p] = x
        take, skip = NEG, NEG                    # SAI: không có dãy rỗng
        for v in a:
            take, skip = (skip if skip > NEG else 0) + v, max(take, skip)
        total += max(take, skip)
    return total

print('sai :', wrong_no_empty(*examples[1][:2]), '| đúng:', Solution().maximumSumSubsequence(list(examples[1][0]), examples[1][1]))

sai : -1 | đúng: 0


**Vì sao sai:** đề cho phép dãy con **rỗng** (tổng 0) — khi mọi phần tử âm, đáp án là 0 chứ không phải phần tử lớn nhất.

### Lỗi 2 — Ghép hai đoạn bằng cách cộng thẳng `L[1][1] + R[1][1]`

In [14]:
def wrong_merge(nums, queries):
    def bad(L, R):
        return (L[0] + R[0], L[1] + R[1], L[2] + R[2], L[3] + R[3])       # SAI: bỏ qua ràng buộc biên
    a = list(nums); total = 0
    for p, x in queries:
        a[p] = x
        total += reduce(bad, map(leaf, a))[3]
    return total

args = ([1, 2, 3], [[0, 1]])
print('sai :', wrong_merge(*args), '| đúng:', Solution().maximumSumSubsequence(list(args[0]), args[1]))

sai : 6 | đúng: 4


**Vì sao sai:** cộng thẳng cho phép chọn cả phần tử cuối của `L` và đầu của `R` — hai phần tử **kề nhau** — nên thực chất cộng **mọi** phần tử dương: `[1, 2, 3]` cho `6` thay vì `1 + 3 = 4`. (Với ví dụ 1 của đề, tình cờ các phần tử dương không kề nhau nên lỗi không lộ ra — cần test riêng.)

### Lỗi 3 — Chỉ trả đáp án của truy vấn cuối

In [15]:
def wrong_last_only(nums, queries):
    a = list(nums)
    for p, x in queries:
        a[p] = x
    return robber(a) % MOD                                                # SAI

print('sai :', wrong_last_only(*examples[0][:2]), '| đúng:', Solution().maximumSumSubsequence(list(examples[0][0]), examples[0][1]))

sai : 9 | đúng: 21


**Vì sao sai:** đề yêu cầu **tổng** đáp án của mọi truy vấn.

### Lỗi 4 — Tính đáp án **trước** khi gán `nums[pos] = x`

In [16]:
def wrong_answer_before_update(nums, queries):
    a = list(nums); total = 0
    for p, x in queries:
        total += robber(a)                                                # SAI: phải gán trước
        a[p] = x
    return total % MOD

print('sai :', wrong_answer_before_update(*examples[0][:2]), '| đúng:', Solution().maximumSumSubsequence(list(examples[0][0]), examples[0][1]))

sai : 24 | đúng: 21


**Vì sao sai:** đề: "**trước tiên** gán, **sau đó** tính". Đảo thứ tự cho `12 + 12 = 24` thay vì `12 + 9 = 21`.

**Ghi chú về thứ tự ghép:** với 4 trạng thái biên (và cả dạng ma trận), bài toán đối xứng khi đảo mảng nên ghép ngược thứ tự *tình cờ* vẫn ra đúng đáp án toàn cục — nhưng sẽ sai ngay khi cần truy vấn đoạn hoặc thông tin biên (bài tập 3, 5). Luôn ghép trái → phải.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "DP tuyến tính + cập nhật điểm + hỏi toàn cục" ⇒ segment tree trên **trạng thái biên** hoặc **ma trận chuyển trạng thái** (max-plus / min-plus).
- Số trạng thái biên = (số trạng thái DP)² — ở đây `2 × 2`.
- Python: tách 4 mảng song song nhanh hơn mảng tuple.

## 15. Biến thể của bài

1. **LC 198 House Robber** (tĩnh), **LC 213** (vòng: xét hai trường hợp đầu/cuối).
2. Truy vấn trên **đoạn** `[l, r]` ⇒ ghép các nút phủ đoạn (giữ thứ tự).
3. Ràng buộc "không chọn hai phần tử cách nhau < `d`" ⇒ trạng thái biên `(d × d)`.
4. Mảng vòng động ⇒ đáp án `max(v[0][1], v[1][0])` của gốc (không được chọn cả đầu lẫn cuối) — xem ô dưới.

In [17]:
# Biến thể 4: mảng vòng (phần tử đầu và cuối cũng kề nhau)
def circular_answer(a):
    if len(a) == 1:
        return max(0, a[0])
    root = reduce(merge, map(leaf, a))
    return max(root[1], root[2])

def circular_brute(a):
    n = len(a); best = 0
    for mask in range(1 << n):
        if n > 1 and any(mask >> i & 1 and mask >> ((i + 1) % n) & 1 for i in range(n)):
            continue
        best = max(best, sum(a[i] for i in range(n) if mask >> i & 1))
    return best

for _ in range(300):
    a = [rng.randint(-5, 9) for _ in range(rng.randint(1, 10))]
    assert circular_answer(a) == circular_brute(a)
assert circular_answer([2, 3, 2]) == 3 and circular_answer([1, 2, 3, 1]) == 4      # ví dụ LC 213
print('Biến thể vòng (LC 213) từ 4 trạng thái biên: khớp brute.')

Biến thể vòng (LC 213) từ 4 trạng thái biên: khớp brute.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Mỗi truy vấn | Ghi chú |
|---|---|---|---|
| 1 | House Robber lại | `O(n)` | Oracle |
| 2 | Segment tree 4 trạng thái | `O(log n)` | **Nộp bài** |
| 3 | Ma trận (max, +) | `O(log n)` | Tổng quát cho mọi DP tuyến tính |
| 4 | Chia căn | `O(√n)` | Đơn giản, chậm hơn |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `nums = [1, 2, 3]`, `queries = [[1, 5]]`.

**Bài 2 (tính tay):** `nums = [2, 2, 2, 2]`, `queries = [[0, -1], [3, 6]]`.

**Bài 3 (điền TODO):** truy vấn đoạn: sau các cập nhật, trả lời `[l, r]` = House Robber trên `nums[l..r]`.

**Bài 4 (tự viết lại):** phiên bản "khoảng cách tối thiểu 3" (không chọn hai phần tử có chỉ số cách nhau `< 3`) với cập nhật điểm.

**Bài 5 (mini challenge):** phiên bản **mảng vòng** động (LC 213 + cập nhật).

In [18]:
def range_query_todo(nums, ops):
    # TODO
    raise NotImplementedError

def gap3_todo(nums, queries):
    # TODO
    raise NotImplementedError

def circular_dynamic_todo(nums, queries):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `[1, 5, 3]`. (2) `5` hay `1 + 3 = 4`. (3) `5`.

**Bài 2.** (1) `[-1, 2, 2, 2]` ⇒ `4`. (2) `[-1, 2, 2, 6]` ⇒ `2 + 6 = 8`. (3) Tổng `12`.

**Bài 3.** (1) Truy vấn segment tree theo đoạn, thu các nút trái → phải. (2) Ghép theo thứ tự. (3) Lấy `v11`.

**Bài 4.** (1) Dùng ma trận chuyển trạng thái với trạng thái "khoảng cách từ lần chọn cuối" `∈ {0, 1, 2+}`. (2) Ma trận `3 × 3` (max, +). (3) Segment tree như cách 3.

**Bài 5.** (1) Segment tree 4 trạng thái như cũ. (2) Đáp án `max(v01, v10)` (cấm đầu hoặc cấm cuối). (3) `n = 1` riêng.

## 19. Đáp án tham khảo cho bài tập

In [19]:
# Bài 1, 2, 3
assert Solution().maximumSumSubsequence([1, 2, 3], [[1, 5]]) == 5
assert Solution().maximumSumSubsequence([2, 2, 2, 2], [[0, -1], [3, 6]]) == 12

def range_query(nums, ops):
    n = len(nums); size = 1
    while size < n:
        size *= 2
    ID = None
    T = [(0, 0, 0, 0)] * (2 * size)
    for i, x in enumerate(nums):
        T[size + i] = leaf(x)
    for i in range(size - 1, 0, -1):
        T[i] = merge(T[2 * i], T[2 * i + 1])
    res = []
    for op in ops:
        if op[0] == 'set':
            _, p, x = op; i = size + p; T[i] = leaf(x); i //= 2
            while i:
                T[i] = merge(T[2 * i], T[2 * i + 1]); i //= 2
        else:
            _, l, r = op; left = []; right = []; l += size; r += size + 1
            while l < r:
                if l & 1: left.append(T[l]); l += 1
                if r & 1: r -= 1; right.append(T[r])
                l //= 2; r //= 2
            res.append(reduce(merge, left + right[::-1])[3])
    return res

for _ in range(200):
    n = rng.randint(1, 10); a = [rng.randint(-5, 9) for _ in range(n)]; b = list(a); ops = []; exp = []
    for _ in range(rng.randint(1, 10)):
        if rng.random() < 0.5:
            p, x = rng.randint(0, n - 1), rng.randint(-5, 9); ops.append(('set', p, x)); b[p] = x
        else:
            l = rng.randint(0, n - 1); r = rng.randint(l, n - 1); ops.append(('ask', l, r)); exp.append(robber(b[l:r + 1]))
    assert range_query(a, ops) == exp
print('Bài 1: 5 | Bài 2: 12 | Bài 3: truy vấn đoạn khớp House Robber trên đoạn.')

Bài 1: 5 | Bài 2: 12 | Bài 3: truy vấn đoạn khớp House Robber trên đoạn.


In [20]:
# Bài 4, 5
def gap3(nums, queries):
    # trạng thái s = số bước kể từ lần chọn cuối, cắt ở 2 ("đã đủ xa"); s=0: vừa chọn
    def M(x):
        R = [[NEG] * 3 for _ in range(3)]          # R[new][old]
        R[0][2] = x                                # chỉ chọn khi đã đủ xa
        R[1][0] = 0
        R[2][1] = 0; R[2][2] = 0
        return R
    def mm(P, Q):
        return [[max(P[i][k] + Q[k][j] for k in range(3)) for j in range(3)] for i in range(3)]
    n = len(nums); size = 1
    while size < n:
        size *= 2
    I = [[0 if i == j else NEG for j in range(3)] for i in range(3)]
    T = [I] * (2 * size)
    for i, x in enumerate(nums):
        T[size + i] = M(x)
    for i in range(size - 1, 0, -1):
        T[i] = mm(T[2 * i + 1], T[2 * i])
    total = 0
    for p, x in queries:
        i = size + p; T[i] = M(x); i //= 2
        while i:
            T[i] = mm(T[2 * i + 1], T[2 * i]); i //= 2
        total += max(0, max(T[1][s][2] for s in range(3)))   # khởi đầu "đủ xa" (s = 2)
    return total

def gap3_brute(nums, queries):
    a = list(nums); total = 0; n = len(a)
    for p, x in queries:
        a[p] = x; best = 0
        for mask in range(1 << n):
            idx = [i for i in range(n) if mask >> i & 1]
            if all(idx[t + 1] - idx[t] >= 3 for t in range(len(idx) - 1)):
                best = max(best, sum(a[i] for i in idx))
        total += best
    return total

def circular_dynamic(nums, queries):
    a = list(nums); total = 0
    for p, x in queries:
        a[p] = x; total += circular_answer(a)      # (bản đơn giản; bản nhanh dùng segment tree 4 trạng thái)
    return total

for _ in range(120):
    n = rng.randint(1, 9); a = [rng.randint(-4, 9) for _ in range(n)]
    q = [[rng.randint(0, n - 1), rng.randint(-4, 9)] for _ in range(rng.randint(1, 5))]
    assert gap3(a, q) == gap3_brute(a, q), (a, q)
    b = list(a); exp = 0
    for p, x in q:
        b[p] = x; exp += circular_brute(b)
    assert circular_dynamic(a, q) == exp
print('Bài 4 (khoảng cách ≥ 3, ma trận 3×3) và Bài 5 (vòng động) khớp brute.')

Bài 4 (khoảng cách ≥ 3, ma trận 3×3) và Bài 5 (vòng động) khớp brute.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
leaf(x)  = (0, 0, 0, max(0, x))                       # (v00, v01, v10, v11)
merge(L, R)[a][b] = max(L[a][1] + R[0][b], L[a][0] + R[1][b])
cập nhật lá -> pull các tổ tiên (trái rồi phải); ans += root.v11
```

**Checklist:**

- [ ] Cho phép dãy rỗng (`max(0, x)` ở lá)?
- [ ] Ghép **cấm** đồng thời cuối trái và đầu phải?
- [ ] Ghép đúng thứ tự trái → phải?
- [ ] Cộng **tổng** đáp án các truy vấn, mod ở cuối?